<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：使用预训练 LLM 生成文本

本笔记中使用的包：

In [5]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目所需的所有依赖

from importlib.metadata import version  # 从标准库导入用于查询已安装包版本的函数
used_libraries = [  # 列出要检查版本的库
    "reasoning_from_scratch",  # 本书/项目的核心库
    "torch",  # PyTorch 深度学习框架
    "tokenizers"  # Hugging Face 的分词库
]

for lib in used_libraries:  # 遍历每个库名
    print(f"{lib} version: {version(lib)}")  # 查询并打印对应库的已安装版本


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
tokenizers version: 0.22.1


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F01_raschka.webp?1" width="500px">

&nbsp;
## 2.1 大型语言模型文本生成入门

- 本节不含代码
- LLM 如何生成文本？
- 本章是设置章节：配置贯穿全书的编码环境和 LLM
- 同时编写将在后续章节使用和扩展的文本生成函数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F02_raschka.webp?1" width="300px">

- LLM（以及一般的神经网络）流程图通常自上而下绘制并阅读。

&nbsp;
## 2.2 设置编码环境

- 如果你在阅读这本书，很可能已经用 Python 开发过
- 如果你已经拥有 Python 3.10 或更高版本的环境，用 `pip` 安装依赖是最简单的方式：

In [2]:
#!pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt

- 在本章中，也可以手动安装这些依赖：

In [3]:
#!pip install torch>=2.7.1 tokenizers>=0.21.2

- 我更推荐使用广受推崇的 [uv](https://docs.astral.sh/uv/) Python 包与项目管理器
- 安装 `uv` 时,请依照官方站点的操作系统安装指南进行: https://docs.astral.sh/uv/getting-started/installation/
- 接着，克隆 GitHub 仓库：

In [4]:
#!git clone --depth 1 https://github.com/rasbt/reasoning-from-scratch.git

- 如果没有安装 `git` 也可以从 Manning 网站手动下载源码，或者直接点击: https://github.com/rasbt/reasoning-from-scratch/archive/refs/heads/main.zip (下载后再解压)


- 在终端里切到  `reasoning-from-scratch` 文件夹。
- 执行  `uv run jupyter lab` 启动 JupyterLab，并打开一个空白 notebook 或本章的 notebook。
- 该命令还会自动在`reasoning-from-scratch` 文件夹内创建本地虚拟环境(通常在  `.venv/`目录)，并依据`pyproject.toml`安装所有依赖。




<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F03_raschka.webp?1" width="500px">

- 如需更多安装细节或其它选项，可参考 [../02_setup-tips/python-instructions.md](../02_setup-tips/python-instructions.md) 


&nbsp;
## 2.3 了解硬件需求与建议


- 如果你刚接触 PyTorch，推荐先阅读我的教程[PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) 
- 如果你按照上一节操作过，PyTorch 已经安装好了。
- 手动检查一下当前 PyTorch 是否支持 GPU，并确认你的机器支持哪些:


In [6]:
import torch

print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print("CUDA GPU")
elif torch.mps.is_available():
    print("Apple Silicon GPU")
else:
    print("Only CPU")

PyTorch version 2.9.0+cu126
CUDA GPU


- 不同章节会自动选择可用的 NVIDIA GPU；如果没有，就落到 CPU（或本章建议的 Apple Silicon GPU）。
- 第 2–4 章在 CPU 上也能在可接受时间内跑完。
- 第 5–7 章在 CPU 上会非常慢，建议使用带 NVIDIA GPU 的机器（后续章节会详细说明资源需求）。
- 我个人偏好 [Lightning AI Studio](https://lightning.ai/)，注册并认证后有免费算力；另外 [Google Colab](https://colab.research.google.com/) 也是不错的选择。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F04_raschka.webp" width="500px">

- 如需云端算力推荐，可参阅  [../02_setup-tips/gpu-instructions.md](../02_setup-tips/gpu-instructions.md) 
- 目前暂时还不用 GPU，前几章在无 GPU 的硬件上就能顺利运行。

&nbsp;
## 2.4 为 LLM 准备输入文本


- 本节我们学习如何使用 tokenizer，将输入文本编码为 token ID 形式，以便送入 LLM。
- 同时也用 tokenizer 把 LLM 的输出解码回人类可读的文本。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F05_raschka.webp?1" width="500px">

- 如前所述，本书不会从零实现 LLM 和 tokenizer，而是专注在已有 LLM/tokenizer 之上，从零构建推理方法。
- 我们会在下一节加载一个预训练 LLM，这一节先加载与之配套的 tokenizer。
- 我准备了 `reasoning_from_scratch` Python 包，提供基础 LLM 及其 tokenizer，底层借助[`tokenizers`](https://github.com/huggingface/tokenizers) 库实现。
- 这个包 `reasoning_from_scratch` 包含在本书的配套代码中，按照第 2.2 节的说明已经安装好了。


- 接下来我们下载 tokenizer 文件（这是 Qwen3 基础版 LLM 所用的 tokenizer，更多细节在下一节介绍）：


In [8]:
from reasoning_from_scratch.qwen3 import download_qwen3_small  # 从项目的 qwen3 模块导入下载小型 Qwen3 模型的函数

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")  # 下载基础版模型只包含分词器，并保存到本地目录 qwen3


- 现在可以把 tokenizer 文件中的配置加载到 `Qwen3Tokenizer` 中:


In [10]:
from pathlib import Path  # 导入标准库 Path 类，用于跨平台处理文件路径
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer  # 从项目的 qwen3 模块导入分词器类

tokenizer_path = Path("qwen3") / "tokenizer-base.json"  # 构建指向已下载分词器文件的路径 qwen3/tokenizer-base.json
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 加载该分词器文件，创建 Qwen3Tokenizer 实例


- 由于我们尚未加载 LLM，本节先做一次更简单的往返：把文本编码成 token ID，再把这些 ID 解码回字符串表示：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F06_raschka.webp" width="500px">

In [ ]:
prompt = "Explain large language models."  # 要分词的输入字符串
input_token_ids_list = tokenizer.encode(prompt)  # 使用加载的分词器将文本编码为 token ID 列表

In [13]:
for i in input_token_ids_list:  # 遍历编码后的每个 token ID
    print(f"{i} --> {tokenizer.decode([i])}")  # 打印 ID 及其解码回的对应 token 字符


840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [14]:
text = tokenizer.decode(input_token_ids_list)  # 将 token ID 列表解码回文本
print(text)  # 输出解码得到的字符串

Explain large language models.


- 对于 `Qwen3Tokenizer`，词表大小大约是 151,000 个唯一 token。

- 更多关于分词的资料：
  - [Build a Large Language Model (from Scratch)](https://mng.bz/M96o) 第 2 章
  - [Implementing A Byte Pair Encoding (BPE) Tokenizer From Scratch](https://sebastianraschka.com/blog/2025/bpe-from-scratch.html)


&nbsp;
## 2.5 Loading pre-trained models


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F07_raschka.webp" width="500px">

- 如上一节加载 tokenizer 时提到的，本书选用 Qwen3 0.6B 作为基础模型；在对比多种开源权重后，选择 Qwen3 的原因是：
  - 截至写作时，Qwen3 在建模性能上是开源权重中的领先者；
  - Qwen3 0.6B 的显存占用比 Llama 3 1B 更少；
  - 提供了基础版（我们以它为底开发推理方法）和官方推理版，后者可作为对照参考。

- 注意 “Qwen3” 的正式写法没有空格，而 “Llama 3” 有空格。
- 本书秉承 “from-scratch” 的精神，使用我基于纯 PyTorch 重写的 Qwen3，不依赖任何外部 LLM 库，并且与原版 Qwen3 权重兼容。
- 不过本书不会展开讲解这份 Qwen3 实现(那又会是另一本书，类似我的 [Build A Large Language Model (From Scratch)](https://github.com/rasbt/LLMs-from-scratch),而是专注于在这个基础模型上实现推理方法。
- Qwen3 模型代码见附录 C。
- 附录 D 介绍如何加载推理版和更大的 Qwen3 模型。
- 更多细节可参考 Qwen3 的 [GitHub repository](https://github.com/QwenLM/Qwen3) 和 [technical report](https://arxiv.org/abs/2505.09388) 


- 这个模型刻意做得较小（但仍然很强大），以便在普通消费级硬件上运行。
- 它能在 CPU、NVIDIA GPU（"cuda"）、Apple Silicon GPU（"mps"）和 Intel GPU（"xpu"）上正常运行，本章后面会进一步讨论各种硬件的性能差异。


In [ ]:
def get_device(enable_tensor_cores=True):  # 选择可用计算设备，可选开启 Tensor Core/TF32
    if torch.cuda.is_available():  # 优先检查 NVIDIA CUDA GPU
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")
        
        if enable_tensor_cores:  # 根据 PyTorch 版本开启 TF32/Tensor Core 支持
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):  # 新版接口
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:  # 旧版接口
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():  # 其次检查 Apple Silicon GPU (Metal)
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():  # 再检查 Intel GPU (XPU)
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:  # 都不可用则退回 CPU
        device = torch.device("cpu")
        print("Using CPU")

    return device  # 返回选定的 device 对象

device = get_device()  # 获取当前可用的计算设备


Using NVIDIA CUDA GPU


- 建议首次体验先在 "cpu" 上运行，所以下面的代码把设备写死为 CPU：

In [16]:
# 初次运行推荐用 CPU，减少依赖 GPU 环境的变数
device = torch.device("cpu")  # 显式创建 CPU 设备对象

- 接着下载预训练权重文件，大小大约 1.5 GB：

In [18]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")  # 下载基础版 Qwen3 模型（包含权重和分词器），保存到 qwen3 目录

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


- 下图展示了我们加载的 Qwen3 0.6B 模型的架构供熟悉 LLM 结构的读者参考，但请注意，本书并不要求理解这些细节，因为我们不会修改模型本身，而是后续在其上叠加推理方法。


- 我在本仓库内的 [reasoning-from-scratch](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/qwen3.py) Python 包里，从零实现了 Qwen3 的模型架构；源码也收录在附录 C。不过这只是给好奇的读者作为附加材料，理解这些内部细节并非阅读本书其它内容的必要条件。


In [19]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B  # 导入 Qwen3 模型类和 0.6B 配置
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"  # 构建已下载模型权重文件的路径

model = Qwen3Model(QWEN_CONFIG_06_B)  # 按 0.6B 配置初始化模型实例（随机权重）
model.load_state_dict(torch.load(model_path))  # 从本地文件加载训练好的权重
model.to(device)  # 将模型移动到前面选定的计算设备


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F08_raschka.webp" width="300px">

&nbsp;
## 2.6 理解 LLM 顺序生成文本的过程

- 本节会编写一个简单的封装函数，方便调用 LLM 来生成文本（第 4 章会在此基础上扩展更多功能）。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F09_raschka.webp?1" width="500px">

- LLM 是一次生成一个词：

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F10_raschka.webp?2" width="500px">

- 上图只是简化示意，只显示了新生成的词；下面的图放大了第一轮迭代的细节。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp" width="3b00px">

In [21]:
example = torch.tensor([1, 2, 3])  # 创建包含 1,2,3 的 1D 张量
print(example)  # 打印原始张量
print(example.unsqueeze(0))  # 在最前面新增一个维度，形状从 (3,) 变为 (1,3)

tensor([1, 2, 3])
tensor([[1, 2, 3]])


In [22]:
example = torch.tensor([[1, 2, 3]])  # 创建形状为 (1, 3) 的二维张量
print(example)  # 打印原始张量
print(example.squeeze(0))  # 去掉第 0 维的长度 1，形状变为 (3,)

tensor([[1, 2, 3]])
tensor([1, 2, 3])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F11_raschka.webp?2" width="300px">

In [ ]:
prompt = "Explain large language models."  # 要编码的输入文本
input_token_ids_list = tokenizer.encode(prompt)  # 文本转为 token ID 列表
print(f"Number of input tokens: {len(input_token_ids_list)}")  # 输出输入 token 数

input_tensor = torch.tensor(input_token_ids_list)  # 将 ID 列表变为 1D 张量
input_tensor_fmt = input_tensor.unsqueeze(0).to(device)  # 在前面加 batch 维并移动到目标设备

output_tensor = model(input_tensor_fmt)  # 前向推理得到模型输出张量
output_tensor_fmt = output_tensor.squeeze(0)  # 去掉 batch 维恢复为 (seq_len, hidden_size)
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")  # 打印输出张量形状

Number of input tokens: 6
Formatted Output tensor shape: torch.Size([6, 151936])


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F12_raschka.webp" width="500px">

In [26]:
last_token = output_tensor_fmt[-1].detach()  # 取输出序列最后一个 token 的隐藏向量
print(last_token)  # 打印该向量

tensor([ 7.2500,  1.8594,  7.8125,  ..., -2.4844, -2.4844, -2.4844],
       dtype=torch.bfloat16)


In [27]:
print(last_token.argmax(dim=-1, keepdim=True))  # 在最后一维取最大值索引（保留维度），并打印结果

tensor([20286])


In [28]:
print(tokenizer.decode([20286]))  # 将 token ID 20286 解码为对应字符串并打印

 Large


In [29]:
example = torch.tensor([-2, 1, 3, 1])  # 创建包含负数和正数的 1D 张量
print(torch.max(example))  # 打印最大值（此处为 3）
print(torch.argmax(example))  # 打印最大值的索引位置（此处为索引 2）

tensor(3)
tensor(2)


&nbsp;
## 2.7 编写一个最简文本生成函数

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F13_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F14_raschka.webp?2" width="500px">

- `generate_text_basic`  函数就实现了上述顺序生成文本的流程：

In [30]:
@torch.inference_mode()  # 关闭梯度计算以加速推理并节省显存
def generate_text_basic(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):
    input_length = token_ids.shape[1]  # 记录原始输入序列长度
    model.eval()  # 切换到评估模式，禁用 dropout 等训练行为

    for _ in range(max_new_tokens):  # 逐个生成新 token，最多生成指定数量
        out = model(token_ids)[:, -1]  # 前向推理并取每个序列的最后位置 logits
        next_token = torch.argmax(out, dim=-1, keepdim=True)  # 选取概率最高的下一个 token

        # Stop if all sequences in the batch have generated EOS
        if (eos_token_id is not None
                and next_token.item() == eos_token_id):  # 若设置了 EOS 且当前生成了 EOS，则提前停止
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)  # 将生成的 token 追加到序列末尾
    return token_ids[:, input_length:]  # 返回新增生成的部分（不含原输入）

- 我们先用它来回答提示 “Explain large language models in 2 sentences.”，生成 100 个 token，看看工作流程（推理相关内容会在后续章节展开）。
- 下面的代码会比较慢，大约需要 1–3 分钟，取决于你的机器（后面章节会优化性能）。


In [31]:
prompt = "Explain large language models in a single sentence."  # 要生成续写的提示文本
input_token_ids_tensor = torch.tensor(  # 将提示编码并放到目标设备上
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)  # 增加 batch 维，形状 (1, seq_len)

max_new_tokens = 100  # 最多生成 100 个新 token
output_token_ids_tensor = generate_text_basic(  # 使用贪心生成新 token
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
)
output_text = tokenizer.decode(  # 解码生成的 token 序列为文本
    output_token_ids_tensor.squeeze(0).tolist()
)
print(output_text)  # 输出生成的文本


 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and symbols, which are used to convey meaning and express thoughts and ideas. The evolution of language has


- 可以看到 LLM 基本遵循了指令，但在 `<|endoftext|>`之后回复开始跑题或胡言乱语；这个 token 是训练时用来分隔不同文档的分隔符。
- 实际使用 LLM 时，我们希望它在遇到该 token 后立刻停止生成。

In [32]:
print(tokenizer.encode("<|endoftext|>"))  # 将特殊 EOS 标记编码为 token ID 列表并打印

[151643]


- 为方便使用，end-of-sequence（eos）这个 token 的 ID 被存成 tokenizer 的属性:

In [33]:
print(tokenizer.eos_token_id)  # 打印分词器的 EOS（结束标记）对应的 token ID

151643


- 我们可以利用这个 ID 来告诉 LLM（准确地说是 `generate_text_basic` 函数）什么时候停止生成文本。

In [34]:
output_token_ids_tensor = generate_text_basic(  # 生成续写，指定 EOS 以遇到结束标记时停止
    model=model,  # 已加载到设备的 Qwen3 模型
    token_ids=input_token_ids_tensor,  # 编码后的提示 token 序列（含 batch 维）
    max_new_tokens=max_new_tokens,  # 最多生成的 token 数
    eos_token_id=tokenizer.eos_token_id  # 结束标记的 token ID
)

output_text = tokenizer.decode(  # 将生成的 token 序列解码为文本
    output_token_ids_tensor.squeeze(0).tolist()
)
print(output_text)  # 打印生成的文本

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


- 上面那段回答是在 CPU 上运行时得到的，具体输出可能会因为设备不同而略有差异。

- 在结束本节并介绍如何提速之前，先实现一个简单的基准函数，用来记录计算性能。

In [35]:
def generate_stats(output_token_ids, tokenizer, start_time,
                   end_time, print_tokens=True):  # 计算并打印生成统计信息，可选打印生成文本
    total_time = end_time - start_time  # 总耗时
    print(f"Time: {total_time:.2f} sec")  # 打印用时
    print(f"{int(output_token_ids.numel() / total_time)} tokens/sec")  # 打印生成速度（tokens 每秒）

    for name, backend in (("CUDA", getattr(torch, "cuda", None)),
                          ("XPU", getattr(torch, "xpu", None))):  # 遍历 CUDA/XPU 后端（若存在）
        if backend is not None and backend.is_available():  # 后端存在且可用时
            max_mem_bytes = backend.max_memory_allocated()  # 获取峰值显存占用（字节）
            max_mem_gb = max_mem_bytes / (1024 ** 3)  # 转换为 GB
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")  # 打印峰值显存
            backend.reset_peak_memory_stats()  # 重置峰值显存统计

    if print_tokens:  # 如需输出生成的文本
        output_text = tokenizer.decode(output_token_ids.squeeze(0).tolist())  # 去掉 batch 维并解码为字符串
        print(f"\n{output_text}")  # 打印生成文本

In [36]:
import time  # 导入标准库 time，用于计时

start_time = time.time()  # 记录开始时间
output_token_ids_tensor = generate_text_basic(  # 调用贪心生成函数
    model=model,  # 已加载的模型
    token_ids=input_token_ids_tensor,  # 编码后的输入 token 序列
    max_new_tokens=max_new_tokens,  # 最大生成 token 数
    eos_token_id=tokenizer.eos_token_id  # 结束标记 ID，遇到即停止
)
end_time = time.time()  # 记录结束时间

generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 计算并打印耗时、速率等统计信息


Time: 22.04 sec
1 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


&nbsp;
## 2.8 通过 KV 缓存加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F15_raschka.webp?2" width="500px">

- 本书的代码更重视可读性，真正要优化可以单独写一本书。
- 这里我们介绍一个工程技巧 “KV 缓存”（KV 指 LLM 注意力机制里的 key 和 value）。
- 不熟悉这些术语也没关系，只要知道可以缓存每次迭代会重复使用的中间值，就能显著提速。


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F16_raschka.webp" width="500px">

- 若想深入了解 KV 缓存的原理，可参考我的文章 [Understanding and Coding the KV Cache in LLMs from Scratch](https://magazine.sebastianraschka.com/p/coding-the-kv-cache-in-llms) 
- 下方给出了加入 KV 缓存后的 `generate_text_basic` 改写版本。


In [37]:
from reasoning_from_scratch.qwen3 import KVCache  # 导入用于存储注意力键值对的缓存类

@torch.inference_mode()  # 关闭梯度计算，加速推理
def generate_text_basic_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None
):

    input_length = token_ids.shape[1]  # 记录原始输入序列长度
    model.eval()  # 切换到评估模式
    cache = KVCache(n_layers=model.cfg["n_layers"])  # 为每一层初始化 KV 缓存
    model.reset_kv_cache()  # 清空模型内部的 KV 缓存状态

    out = model(token_ids, cache=cache)[:, -1]  # 首次前向传播并获取最后位置 logits
    for _ in range(max_new_tokens):  # 逐 token 生成，最多生成指定数量
        next_token = torch.argmax(out, dim=-1, keepdim=True)  # 选取概率最高的下一个 token

        if (eos_token_id is not None 
               and next_token.item() == eos_token_id):  # 若生成 EOS，提前结束
            break

        token_ids = torch.cat([token_ids, next_token], dim=1)  # 将新 token 追加到序列
        out = model(next_token, cache=cache)[:, -1]  # 使用缓存增量前向，获取当前最后位置 logits

    return token_ids[:, input_length:]  # 只返回新增生成的 token 部分


- 用法跟之前几乎一样：

In [38]:
start_time = time.time()  # 记录开始时间

output_token_ids_tensor = generate_text_basic_cache(  # 使用带 KV 缓存的生成函数
    model=model,  # 已加载模型
    token_ids=input_token_ids_tensor,  # 编码后的提示序列
    max_new_tokens=max_new_tokens,  # 最大生成 token 数
    eos_token_id=tokenizer.eos_token_id,  # 结束标记 ID
)
end_time = time.time()  # 记录结束时间

generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 打印耗时、速率和显存等统计


Time: 3.00 sec
13 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.


- 可以看到速度快了好几个数量级（在 Mac Mini M4 CPU 上从 4 tokens/sec 提升到 28 tokens/sec）。

&nbsp;
## 2.9 通过 PyTorch 模型编译加速推理

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch02/CH02_F17_raschka.webp?2" width="500px">

- 还有一种大幅加速推理（文本生成）的技巧：使用`torch.compile`
- 用法很简单，直接对模型调用 `torch.compile` 即可( 更多配置选项见[the documentation](https://docs.pytorch.org/docs/stable/torch.compiler.html) )


In [39]:
major, minor = map(int, torch.__version__.split(".")[:2])  # 解析当前 PyTorch 主次版本号
if (major, minor) >= (2, 8):  # 仅在 PyTorch 2.8+ 执行以下设置
    # This avoids retriggering model recompilations 
    # in PyTorch 2.8 and newer
    # if the model contains code like self.pos = self.pos + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True  # 允许 nn.Module 内的不定长整型，避免重复编译

model_compiled = torch.compile(model)  # 使用 torch.compile 对模型进行编译以加速推理

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer

---

**Windows 注意事项 1**

- 在 Windows 上编译较为麻烦。
- `torch.compile()` 依赖 Inductor，它会 JIT 编译 kernel，需要可用的 C/C++ 工具链。
- 对 CUDA 而言，Inductor 还依赖 Triton，可通过社区包 `triton-windows` 获取。
  - 若提示 `cl not found`, [安装带 “C++ Workload” 的 Visual Studio Build Tools](https://learn.microsoft.com/en-us/cpp/build/vscpp-step-0-installation?view=msvc-170) 并在 “x64 Native Tools” 命令行里运行 Python。
  - 若 CUDA 情况下出现 triton not found，安装 `triton-windows` (例如, `uv pip install "triton-windows<3.4"`).
- 对 CPU，读者建议参考 [PyTorch Inductor guide for Windows](https://docs.pytorch.org/tutorials/unstable/inductor_windows.html)
  -安装 Visual Studio 2022 时务必添加英文语言包，避免 UTF-8 错误。
  - 另外需要在 “Visual Studio 2022 Developer Command Prompt” 中运行代码，而不是在 notebook。
- 若上述配置太折腾，可以跳过编译；编译纯属可选，示例代码在不编译的情况下也能正常运行。

**Windows 注意事项 2**

- 有读者反馈，在 Windows 默认参数下运行`torch.compile` 没有提速；但使用  `torch.compile(model, mode="max-autotune")`可带来约 2 倍的加速。

---


- 由于第一次运行要做编译和优化，可能比较慢，所以会把生成过程重复多次。
- 先从不使用缓存的版本开始（可能会有些慢，耗时几分钟）。

In [40]:
for i in range(3):  # 连续运行 3 次：第 1 次热身，后两次计时
    start_time = time.time()  # 记录开始时间
    output_token_ids_tensor = generate_text_basic(  # 使用贪心生成文本
        model=model_compiled,  # 已 torch.compile 的模型
        token_ids=input_token_ids_tensor,  # 编码后的输入序列
        max_new_tokens=max_new_tokens,  # 最大生成 token 数
        eos_token_id=tokenizer.eos_token_id  # 结束标记 ID
    )
    end_time = time.time()  # 记录结束时间

    if i == 0:
        print("Warm-up run")  # 首次运行作为热身，不计性能
    else:
        print(f"Timed run {i}:")  # 输出当前计时轮次
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)  # 打印耗时、速率等统计

    print(f"\n{30*'-'}\n")  # 分隔线，便于阅读输出


Warm-up run
Time: 41.66 sec
0 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 1:
Time: 19.85 sec
2 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 2:
Time: 19.90 sec
2 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing art

- 如上所示，速度大约 5 tokens/sec，比之前 4 tokens/sec 只快一点点。
- 接下来看看启用 KV 缓存的版本表现如何。

In [41]:
for i in range(3):  # 连续运行 3 次：第 1 次热身，后两次计时
    start_time = time.time()  # 记录开始时间
    output_token_ids_tensor = generate_text_basic_cache(  # 使用带 KV 缓存的生成函数
        model=model_compiled,  # 已编译的模型
        token_ids=input_token_ids_tensor,  # 编码后的输入序列
        max_new_tokens=max_new_tokens,  # 最大生成 token 数
        eos_token_id=tokenizer.eos_token_id  # 结束标记 ID
    )
    end_time = time.time()  # 记录结束时间

    if i == 0:
        print("Warm-up run")  # 第一次作为热身，不计性能
    else:
        print(f"Timed run {i}:")  # 输出计时轮次
    generate_stats(  # 打印耗时、速率、显存等统计
        output_token_ids_tensor, tokenizer, start_time, end_time
    )

    print(f"\n{30*'-'}\n")  # 分隔线，便于查看输出


Warm-up run
Time: 200.23 sec
0 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 1:
Time: 1.79 sec
22 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

------------------------------

Timed run 2:
Time: 1.77 sec
23 tokens/sec
Max CUDA memory allocated: 0.00 GB

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing ar

- 可以看到编译后带来了显著的 2 倍提速（从 30 tokens/sec 提升到 64 tokens/sec）。
- 下表列出了更多测量结果。

| Model      | Mode              | Hardware             | Tokens/sec    | GPU Memory (VRAM) |
|------------|-------------------|----------------------|---------------|-------------------|
| Qwen3Model | Regular           | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | Regular compiled  | Mac Mini M4 CPU      | 5             | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 CPU      | 29            | -                 |
| Qwen3Model | KV cache compiled | Mac Mini M4 CPU      | 68            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Regular           | Mac Mini M4 GPU      | 27            | -                 |
| Qwen3Model | Regular compiled  | Mac Mini M4 GPU      | 43            | -                 |
| Qwen3Model | KV cache          | Mac Mini M4 GPU      | 41            | -                 |
| Qwen3Model | KV cache compiled | Mac Mini M4 GPU      | 71            | -                 |
|            |                   |                      |               |                   |
| Qwen3Model | Regular           | NVIDIA H100 GPU      | 51            | 1.55 GB           |
| Qwen3Model | Regular compiled  | NVIDIA H100 GPU      | 164           | 1.81 GB           |
| Qwen3Model | KV cache          | NVIDIA H100 GPU      | 48            | 1.52 GB           |
| Qwen3Model | KV cache compiled | NVIDIA H100 GPU      | 141           | 1.81 GB           |
|            |                   |                      |               |                   |
| Qwen3Model | Regular           | NVIDIA DGX Spark GPU | 72            | 1.53 GB           |
| Qwen3Model | Regular compiled  | NVIDIA DGX Spark GPU | 118           | 1.49 GB           |
| Qwen3Model | KV cache          | NVIDIA DGX Spark GPU | 69            | 1.47 GB           |
| Qwen3Model | KV cache compiled | NVIDIA DGX Spark GPU | 107           | 1.47 GB           |


- 上述 NVIDIA DGX Spark 使用的是 GB10（Blackwell）GPU。
- 需要注意，我们所有示例都只用了单条提示（即 batch size = 1）；如果想了解批量推理，请参见附录 E。

&nbsp;
## 总结

- 本节没有代码。